In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score, f1_score
from sklearn.model_selection import GridSearchCV

In [2]:
X_train = pd.read_csv("data/X_train.csv")
X_test = pd.read_csv("data/X_test.csv")
y_train = pd.read_csv("data/y_train.csv").values.ravel()
y_test = pd.read_csv("data/y_test.csv").values.ravel()

In [3]:
rare_classes = ["type", "hover", "scrolldown", "appear", "disappear", "scrollup"]
y_train = pd.Series(y_train).replace(rare_classes, "other").values
y_test = pd.Series(y_test).replace(rare_classes, "other").values

In [4]:
classes = np.sort(np.unique(y_train))
print("Classes:", classes)
print(f"Number of classes: {len(classes)}")

Classes: ['click' 'drag' 'other' 'select' 'zoomin' 'zoomout']
Number of classes: 6


In [5]:
rf = RandomForestClassifier(random_state=42)

param_grid = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [None, 10, 20, 30, 50],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"],
}

grid_search = GridSearchCV(
    rf,
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train, y_train)

# Best results
print("Best parameters:", grid_search.best_params_)
print("Best score:", grid_search.best_score_)

Fitting 5 folds for each of 360 candidates, totalling 1800 fits
[CV] END max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   2.7s
[CV] END max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   2.8s
[CV] END max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   3.1s
[CV] END max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   3.1s
[CV] END max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   3.2s
[CV] END max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=   5.8s
[CV] END max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=   6.4s
[CV] END max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=

In [6]:
best_rf = grid_search.best_estimator_
y_pred = best_rf.predict(X_test)

In [7]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

print("\nMulti-class Random Forest Results:")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f} (weighted)")
print(f"  Recall:    {recall:.4f} (weighted)")
print(f"  F1:        {f1:.4f} (weighted)")

print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))


Multi-class Random Forest Results:
  Accuracy:  0.7480
  Precision: 0.7578 (weighted)
  Recall:    0.7480 (weighted)
  F1:        0.7380 (weighted)

Classification report:
              precision    recall  f1-score   support

       click       0.66      0.75      0.70       291
        drag       0.77      0.80      0.79       426
       other       1.00      0.05      0.10        20
      select       0.67      0.17      0.28        23
      zoomin       0.83      0.81      0.82        70
     zoomout       0.94      0.76      0.84        67

    accuracy                           0.75       897
   macro avg       0.81      0.56      0.59       897
weighted avg       0.76      0.75      0.74       897

